In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import tracksdata as td

from sklearn.model_selection import StratifiedKFold


ROOT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

DATA_ROOT = (
    ROOT
    / "data/raw/competition"
    / "biohub-cell-tracking-during-development"
)

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

train_geffs = sorted(
    TRAIN_DIR.glob("*.geff")
)

test_stems = {
    p.stem
    for p in TEST_DIR.glob("*.zarr")
}

print("Train datasets:", len(train_geffs))
print("Downloadable test datasets:", len(test_stems))

In [ ]:
records = []

for geff_path in train_geffs:

    stem = geff_path.stem
    prefix = stem.split("_")[0]

    graph, metadata = (
        td.graph.IndexedRXGraph.from_geff(
            geff_path
        )
    )

    estimated_nodes = int(
        metadata.extra[
            "estimated_number_of_nodes"
        ]
    )

    t_axis = next(
        axis
        for axis in metadata.axes
        if axis.name == "t"
    )

    n_frames = int(
        t_axis.max
        - t_axis.min
        + 1
    )

    node_ids = graph.node_ids()

    out_degree = np.asarray(
        graph.out_degree(node_ids)
    )

    node_t = (
        graph.node_attrs(
            attr_keys=["t"]
        )["t"]
        .to_numpy()
    )

    records.append({
        "dataset": stem,
        "prefix": prefix,

        "n_frames":
            n_frames,

        "estimated_nodes":
            estimated_nodes,

        "estimated_per_frame":
            estimated_nodes / n_frames,

        "sparse_gt_nodes":
            graph.num_nodes(),

        "sparse_gt_edges":
            graph.num_edges(),

        "sparse_gt_per_frame":
            graph.num_nodes() / n_frames,

        "sparse_fraction":
            graph.num_nodes()
            / estimated_nodes,

        "n_divisions":
            int(
                np.sum(
                    out_degree == 2
                )
            ),

        "annotated_frames":
            int(
                len(
                    np.unique(node_t)
                )
            ),

        "is_downloadable_test_overlap":
            stem in test_stems,
    })


cv_df = pd.DataFrame(records)

cv_df.shape

In [ ]:
cv_df.groupby("prefix").agg(
    n=("dataset", "size"),
    estimated_per_frame_median=(
        "estimated_per_frame",
        "median",
    ),
    sparse_fraction_median=(
        "sparse_fraction",
        "median",
    ),
    divisions_total=(
        "n_divisions",
        "sum",
    ),
    test_overlap=(
        "is_downloadable_test_overlap",
        "sum",
    ),
)

In [ ]:
division_dataset_summary = (
    cv_df
    .groupby("prefix")
    .agg(
        n_datasets=(
            "dataset",
            "size",
        ),

        datasets_with_division=(
            "n_divisions",
            lambda x: int(
                (x > 0).sum()
            ),
        ),

        datasets_without_division=(
            "n_divisions",
            lambda x: int(
                (x == 0).sum()
            ),
        ),

        total_divisions=(
            "n_divisions",
            "sum",
        ),

        mean_divisions=(
            "n_divisions",
            "mean",
        ),

        median_divisions=(
            "n_divisions",
            "median",
        ),

        max_divisions=(
            "n_divisions",
            "max",
        ),
    )
)

division_dataset_summary

In [ ]:
cv_df.groupby(
    "prefix"
)["n_divisions"].value_counts().sort_index()

In [ ]:
cv_df[
    [
        "dataset",
        "prefix",
        "estimated_per_frame",
        "sparse_fraction",
        "n_divisions",
        "is_downloadable_test_overlap",
    ]
].sort_values(
    "n_divisions",
    ascending=False,
).head(20)

每个 fold 同时尽量平衡：

prefix+cell density+division events+dataset size
	​


In [ ]:
from sklearn.model_selection import StratifiedKFold


# ---------------------------------------------------------
# 1. Density strata within each prefix
# ---------------------------------------------------------

cv_df = cv_df.copy()

cv_df["density_bin"] = (
    cv_df
    .groupby("prefix")["estimated_per_frame"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            q=5,
            labels=False,
        )
    )
    .astype(int)
)

cv_df["stratum"] = (
    cv_df["prefix"]
    + "_d"
    + cv_df["density_bin"].astype(str)
)

cv_df["has_division"] = (
    cv_df["n_divisions"] > 0
).astype(int)


print(
    cv_df.groupby(
        ["prefix", "density_bin"]
    ).size()
)

In [ ]:
# ---------------------------------------------------------
# 2. Candidate split scoring
# ---------------------------------------------------------

N_FOLDS = 5
N_SEEDS = 5000


def relative_std(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    mean = values.mean()

    if mean == 0:
        return 0.0

    return (
        values.std()
        / mean
    )


def score_fold_assignment(df):

    # Overall fold balance
    fold_summary = (
        df.groupby("fold")
        .agg(
            n_datasets=(
                "dataset",
                "size",
            ),
            estimated_nodes=(
                "estimated_nodes",
                "sum",
            ),
            sparse_gt_edges=(
                "sparse_gt_edges",
                "sum",
            ),
            n_divisions=(
                "n_divisions",
                "sum",
            ),
            datasets_with_division=(
                "has_division",
                "sum",
            ),
        )
        .reindex(
            range(N_FOLDS)
        )
    )

    score = 0.0

    # Most important:
    # division + evaluation-weight balance
    score += (
        4.0
        * relative_std(
            fold_summary["n_divisions"]
        )
    )

    score += (
        2.0
        * relative_std(
            fold_summary[
                "datasets_with_division"
            ]
        )
    )

    score += (
        2.0
        * relative_std(
            fold_summary[
                "sparse_gt_edges"
            ]
        )
    )

    # Dataset / cell-count balance
    score += (
        1.0
        * relative_std(
            fold_summary[
                "estimated_nodes"
            ]
        )
    )

    score += (
        1.0
        * relative_std(
            fold_summary[
                "n_datasets"
            ]
        )
    )

    # Balance divisions independently
    # inside 44b6 and 6bba
    for prefix in [
        "44b6",
        "6bba",
    ]:

        prefix_divisions = (
            df[
                df["prefix"] == prefix
            ]
            .groupby("fold")[
                "n_divisions"
            ]
            .sum()
            .reindex(
                range(N_FOLDS),
                fill_value=0,
            )
        )

        score += (
            2.0
            * relative_std(
                prefix_divisions
            )
        )

    return score

In [ ]:
best_score = np.inf
best_seed = None
best_folds = None

X = np.zeros(
    len(cv_df)
)

y = cv_df[
    "stratum"
].to_numpy()


for seed in range(N_SEEDS):

    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=seed,
    )

    fold_assignment = np.full(
        len(cv_df),
        -1,
        dtype=int,
    )

    for fold, (_, val_idx) in enumerate(
        skf.split(X, y)
    ):
        fold_assignment[
            val_idx
        ] = fold

    candidate = cv_df.copy()

    candidate["fold"] = (
        fold_assignment
    )

    candidate_score = (
        score_fold_assignment(
            candidate
        )
    )

    if candidate_score < best_score:

        best_score = candidate_score
        best_seed = seed
        best_folds = (
            fold_assignment.copy()
        )


cv_df["fold"] = best_folds

print(
    "Best seed:",
    best_seed,
)

print(
    "Balance score:",
    best_score,
)

In [ ]:
fold_summary = (
    cv_df
    .groupby("fold")
    .agg(
        n_datasets=(
            "dataset",
            "size",
        ),

        n_44b6=(
            "prefix",
            lambda x: int(
                (x == "44b6").sum()
            ),
        ),

        n_6bba=(
            "prefix",
            lambda x: int(
                (x == "6bba").sum()
            ),
        ),

        estimated_nodes=(
            "estimated_nodes",
            "sum",
        ),

        sparse_gt_nodes=(
            "sparse_gt_nodes",
            "sum",
        ),

        sparse_gt_edges=(
            "sparse_gt_edges",
            "sum",
        ),

        datasets_with_division=(
            "has_division",
            "sum",
        ),

        n_divisions=(
            "n_divisions",
            "sum",
        ),

        test_overlap=(
            "is_downloadable_test_overlap",
            "sum",
        ),
    )
)

fold_summary

In [ ]:
division_fold_pivot = cv_df.pivot_table(
    index="fold",
    columns="prefix",
    values="n_divisions",
    aggfunc="sum",
    fill_value=0,
)

division_fold_pivot

In [ ]:
import hashlib
import json


CONFIG_DIR = ROOT / "configs"
CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CV_PATH = (
    CONFIG_DIR
    / "cv_manifest.csv"
)


# Keep useful audit information,
# but remove temporary stratification label.
save_columns = [
    "dataset",
    "prefix",
    "fold",

    "n_frames",

    "estimated_nodes",
    "estimated_per_frame",

    "sparse_gt_nodes",
    "sparse_gt_edges",
    "sparse_gt_per_frame",
    "sparse_fraction",

    "annotated_frames",

    "n_divisions",
    "has_division",

    "density_bin",

    "is_downloadable_test_overlap",
]


cv_manifest = (
    cv_df[
        save_columns
    ]
    .sort_values(
        "dataset"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Sanity checks before locking
# ---------------------------------------------------------

assert len(cv_manifest) == 199

assert cv_manifest[
    "dataset"
].is_unique

assert set(
    cv_manifest["fold"]
) == {0, 1, 2, 3, 4}

assert not cv_manifest.isna().any().any()


# Save deterministically
cv_manifest.to_csv(
    CV_PATH,
    index=False,
    lineterminator="\n",
)


# ---------------------------------------------------------
# SHA-256
# ---------------------------------------------------------

cv_bytes = CV_PATH.read_bytes()

cv_sha256 = hashlib.sha256(
    cv_bytes
).hexdigest()


print(
    "Saved:",
    CV_PATH
)

print(
    "Rows:",
    len(cv_manifest)
)

print(
    "SHA256:",
    cv_sha256
)

In [ ]:
from pandas.testing import assert_frame_equal

cv_check = pd.read_csv(
    CV_PATH
)

# Basic structural checks
assert len(cv_check) == 199
assert cv_check["dataset"].is_unique
assert set(cv_check["fold"]) == {0, 1, 2, 3, 4}

# Same columns and row order
cv_check = (
    cv_check
    .sort_values("dataset")
    .reset_index(drop=True)
)

cv_reference = (
    cv_manifest
    .sort_values("dataset")
    .reset_index(drop=True)
)

assert list(cv_check.columns) == list(cv_reference.columns)

# CSV float serialization can introduce tiny round-trip differences.
assert_frame_equal(
    cv_check,
    cv_reference,
    check_exact=False,
    check_dtype=False,
    rtol=1e-12,
    atol=1e-12,
)

print("CV manifest round-trip: OK")

In [ ]:
import hashlib

cv_sha256 = hashlib.sha256(
    CV_PATH.read_bytes()
).hexdigest()

print("CV manifest:", CV_PATH)
print("Rows:", len(cv_check))
print("SHA256:", cv_sha256)
print("Best seed:", best_seed)
print("Balance score:", best_score)

In [ ]:
import json

CV_LOCK_PATH = (
    CONFIG_DIR
    / "cv_manifest.lock.json"
)

cv_lock = {
    "version": 1,
    "n_datasets": 199,
    "n_folds": 5,
    "best_seed": int(best_seed),
    "balance_score": float(best_score),
    "sha256": cv_sha256,
    "rules": [
        "dataset-level split only",
        "prefix-stratified",
        "density-stratified",
        "division-balanced",
        "shared across all model experiments",
    ],
}

CV_LOCK_PATH.write_text(
    json.dumps(
        cv_lock,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(CV_LOCK_PATH.read_text())

best_seed      = 1640
balance_score  = 0.8097901436759866
SHA256         = c30ae9625957c5c1a2b68b8808638bdbce4f71f72432c7e0e31f9629e91b4e0f
datasets       = 199
folds          = 5

In [ ]:
OFFICIAL_SPLITS_PATH = (
    CONFIG_DIR
    / "dataset_splits.json"
)

official_splits = []

for fold in range(5):

    val_stems = (
        cv_manifest.loc[
            cv_manifest["fold"] == fold,
            "dataset",
        ]
        .sort_values()
        .tolist()
    )

    train_stems = (
        cv_manifest.loc[
            cv_manifest["fold"] != fold,
            "dataset",
        ]
        .sort_values()
        .tolist()
    )

    official_splits.append({
        "split": fold,
        "train": train_stems,
        "test": val_stems,
    })


OFFICIAL_SPLITS_PATH.write_text(
    json.dumps(
        official_splits,
        indent=2,
    )
    + "\n"
)

print("Saved:", OFFICIAL_SPLITS_PATH)

for split in official_splits:
    print(
        f"Fold {split['split']}: "
        f"{len(split['train'])} train / "
        f"{len(split['test'])} val"
    )

In [ ]:
loaded_splits = json.loads(
    OFFICIAL_SPLITS_PATH.read_text()
)

for fold_data in loaded_splits:

    fold = fold_data["split"]

    expected_val = set(
        cv_manifest.loc[
            cv_manifest["fold"] == fold,
            "dataset",
        ]
    )

    expected_train = set(
        cv_manifest.loc[
            cv_manifest["fold"] != fold,
            "dataset",
        ]
    )

    assert set(fold_data["test"]) == expected_val
    assert set(fold_data["train"]) == expected_train

    assert (
        set(fold_data["train"])
        & set(fold_data["test"])
    ) == set()

print(
    "Official split JSON matches "
    "locked CV manifest: OK"
)

In [ ]:
import hashlib
import json

loaded_splits = json.loads(
    OFFICIAL_SPLITS_PATH.read_text()
)

for fold_data in loaded_splits:
    print(
        f"Fold {fold_data['split']}: "
        f"{len(fold_data['train'])} train / "
        f"{len(fold_data['test'])} val"
    )

official_splits_sha256 = hashlib.sha256(
    OFFICIAL_SPLITS_PATH.read_bytes()
).hexdigest()

print()
print(
    "Official splits SHA256:",
    official_splits_sha256,
)